# GENIE Task 3 Diffusion

This notebook is the mentor-facing Colab wrapper for the exploratory Task 3 diffusion baseline.
It uses `src/task3_diffusion.py` as the source of truth and does not reimplement the diffusion model inside the notebook.

Active detector channel order: `Tracks`, `ECAL`, `HCAL`.
Task 3 is intentionally exploratory because sparse detector structure is not naturally well modeled as a dense image DDPM.


In [ ]:
# 1. Optional Google Drive mount
from pathlib import Path

USE_GOOGLE_DRIVE = True
DRIVE_MOUNT_POINT = Path('/content/drive')

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount(str(DRIVE_MOUNT_POINT))
        print(f'Mounted Google Drive at {DRIVE_MOUNT_POINT}')
    except ImportError:
        print('google.colab is not available in this environment.')
else:
    print('Skipping Google Drive mount.')


In [ ]:
# 2. Clone repo and install dependencies
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/nitik1998/GENIE_DiffusionLearning.git'
REPO_DIR = Path('/content/GENIE_DiffusionLearning')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'reset', '--hard', 'origin/main'], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Repo ready at: {REPO_DIR}')
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
# 3. Imports and runtime config
from pathlib import Path
import json
import os
import random
import shutil
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, Markdown, display

from src.config import CHANNEL_NAMES
from src.data_utils import load_dataset

SEED = 42

TASK_CONFIG = {
    'exp_name': 'task3_final_diffusion',
    'display_name': 'Task 3 Final Diffusion Run',
    'epochs': 30,
    'batch_size': 0,
    'max_events': None,
    'timesteps': 1000,
    'n_samples': 8,
    'loss_type': 'l1',
}

DATASET_SOURCE = Path('/content/drive/MyDrive/quark-gluon_data-set_n139306.hdf5')
if not DATASET_SOURCE.exists():
    DATASET_SOURCE = Path('/content/quark-gluon_data-set_n139306.hdf5')

RESULTS_ROOT = Path('/content/results_colab')
CHECKPOINT_ROOT = RESULTS_ROOT / 'checkpoints'
DATA_DIR = REPO_DIR / 'data'
DATASET_TARGET = DATA_DIR / 'quark-gluon_data-set_n139306.hdf5'
FINAL_RESULTS_DIR = RESULTS_ROOT / 'task3' / TASK_CONFIG['exp_name']

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR.mkdir(parents=True, exist_ok=True)
for path in [RESULTS_ROOT, CHECKPOINT_ROOT, FINAL_RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if DATASET_SOURCE.exists() and DATASET_SOURCE.resolve() != DATASET_TARGET.resolve():
    shutil.copy2(DATASET_SOURCE, DATASET_TARGET)
elif not DATASET_TARGET.exists():
    raise FileNotFoundError(f'Dataset not found at {DATASET_SOURCE} or {DATASET_TARGET}')

os.environ['GENIE_PROJECT_ROOT'] = str(REPO_DIR)
os.environ['GENIE_DATA_DIR'] = str(DATA_DIR)
os.environ['GENIE_OUTPUT_DIR'] = str(RESULTS_ROOT)
os.environ['GENIE_CHECKPOINT_DIR'] = str(CHECKPOINT_ROOT)

print(f'Python executable: {sys.executable}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device: {torch.cuda.get_device_name(0)}')
print(f"Active experiment: {TASK_CONFIG['exp_name']} ({TASK_CONFIG['display_name']})")

def run_cli(cmd):
    cmd = list(cmd)
    if cmd and Path(cmd[0]).name.startswith('python'):
        cmd.insert(1, '-u')
    print('Running:', ' '.join(map(str, cmd)), flush=True)
    print('Logs stream directly below. Estimate remaining time from the first completed epoch.', flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    subprocess.run(cmd, check=True, env=env)

def show_png(path, width=900):
    path = Path(path)
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f'Missing image: {path}')

def print_metrics(path, title):
    path = Path(path)
    if not path.exists():
        print(f'Missing metrics file: {path}')
        return
    payload = json.loads(path.read_text())
    metrics = payload.get('metrics', payload)
    display(Markdown(f'### {title}'))
    for key, value in metrics.items():
        print(f'{key}: {value}')


In [ ]:
# 4. Data sanity check
sanity_events = 128 if TASK_CONFIG['max_events'] is None else min(TASK_CONFIG['max_events'], 128)
X, y = load_dataset(str(DATA_DIR), max_events=sanity_events)
print('Loaded shape:', X.shape)
print('Labels shape:', y.shape)
print('Channel names:', CHANNEL_NAMES)

fig, axes = plt.subplots(3, 3, figsize=(10, 10), squeeze=False)
for row in range(3):
    for ch in range(3):
        img = X[row, ch]
        vmax = max(np.percentile(img[img > 0], 99.5), 1e-6) if np.any(img > 0) else 1e-6
        axes[row, ch].imshow(img, cmap='hot', vmin=0, vmax=vmax)
        axes[row, ch].axis('off')
        if row == 0:
            axes[row, ch].set_title(CHANNEL_NAMES[ch])
plt.tight_layout()
plt.show()


In [ ]:
# 5. Final Task 3 config
display(Markdown(f"## Active Config: `{TASK_CONFIG['exp_name']}`"))
for key, value in TASK_CONFIG.items():
    print(f'{key}: {value}')


In [ ]:
# 6. Run Task 3 through the official script
cmd = [
    sys.executable,
    'src/task3_diffusion.py',
    '--exp-name', TASK_CONFIG['exp_name'],
    '--epochs', str(TASK_CONFIG['epochs']),
    '--batch-size', str(TASK_CONFIG['batch_size']),
    '--timesteps', str(TASK_CONFIG['timesteps']),
    '--n-samples', str(TASK_CONFIG['n_samples']),
    '--loss-type', TASK_CONFIG['loss_type'],
    '--seed', str(SEED),
]
if TASK_CONFIG.get('max_events') is not None:
    cmd += ['--max-events', str(TASK_CONFIG['max_events'])]
run_cli(cmd)


In [ ]:
# 7. Visualize final outputs inline
display(Markdown(f"## Final Outputs: `{TASK_CONFIG['exp_name']}`"))
show_png(FINAL_RESULTS_DIR / 'original_vs_generated_or_reconstructed.png', width=1200)
show_png(FINAL_RESULTS_DIR / 'training_curves.png', width=900)
print_metrics(FINAL_RESULTS_DIR / 'metrics.json', 'Task 3 Metrics')


In [ ]:
# 8. Package artifacts for download
archive_base = RESULTS_ROOT / TASK_CONFIG['exp_name']
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=str(FINAL_RESULTS_DIR.parent), base_dir=FINAL_RESULTS_DIR.name)
print(f'Created archive: {archive_path}')

try:
    from google.colab import files
    files.download(archive_path)
except Exception as exc:
    print(f'Automatic download skipped: {exc}')
